# Embeddings vs LLMs: Why Does RAG Need Both?

A RAG system usually uses one model to find relevant information and another model to generate the final answer.

I wanted to look at these two steps separately and see what each model is actually doing.

For this experiment, I'll use a small set of support cases, retrieve the most relevant information using embeddings, and then pass that information to an LLM to generate the answer.

In [1]:
import ollama
import numpy as np

## A small knowledge base

I'll use a few support cases as the information available to the system.

First, I want to see if the embedding model can find the right information even when the question is written differently.

In [2]:
documents = [
    "If a user forgets their password, they can reset it using the Forgot Password link on the login page.",

    "Users can update their billing information from Settings > Billing > Payment Method.",

    "If a file upload fails, check that the file is smaller than 10 MB and uses a supported format.",

    "Accounts are temporarily locked after five failed login attempts. The account unlocks automatically after 30 minutes.",

    "Users can enable two-factor authentication from Settings > Security."
]

for i, document in enumerate(documents, start=1):
    print(f"{i}. {document}")

1. If a user forgets their password, they can reset it using the Forgot Password link on the login page.
2. Users can update their billing information from Settings > Billing > Payment Method.
3. If a file upload fails, check that the file is smaller than 10 MB and uses a supported format.
4. Accounts are temporarily locked after five failed login attempts. The account unlocks automatically after 30 minutes.
5. Users can enable two-factor authentication from Settings > Security.


In [3]:
query = "I entered the wrong password several times and now I can't sign in."

print(query)

I entered the wrong password several times and now I can't sign in.


## Turning the text into embeddings

I'll convert the support cases and the question into embeddings using `nomic-embed-text`.

The embedding model doesn't generate an answer. It gives us numerical representations that we can use to compare how closely the texts are related.

In [4]:
document_response = ollama.embed(
    model="nomic-embed-text",
    input=documents
)

document_embeddings = np.array(document_response["embeddings"])

query_response = ollama.embed(
    model="nomic-embed-text",
    input=query
)

query_embedding = np.array(query_response["embeddings"][0])

print("Documents:", document_embeddings.shape)
print("Query:", query_embedding.shape)

Documents: (5, 768)
Query: (768,)


In [5]:
print("First 10 values of the query embedding:")
print(query_embedding[:10])

First 10 values of the query embedding:
[ 0.01651595  0.05405407 -0.1207234   0.02789907  0.05100745  0.0187285
 -0.00202587  0.06033097  0.04662592 -0.03685037]


## Finding the closest match

Now I'll compare the question embedding with each support case.

A higher cosine similarity means the texts are closer in meaning according to the embedding model.

In [6]:
def cosine_similarity(a, b):
    return np.dot(a, b) / (
        np.linalg.norm(a) * np.linalg.norm(b)
    )


similarities = [
    cosine_similarity(query_embedding, document_embedding)
    for document_embedding in document_embeddings
]

results = []

for i, (document, similarity) in enumerate(
    zip(documents, similarities),
    start=1
):
    results.append({
        "Document": i,
        "Similarity": similarity,
        "Text": document
    })

results = sorted(
    results,
    key=lambda x: x["Similarity"],
    reverse=True
)

for result in results:
    print(
        f'Document {result["Document"]} '
        f'({result["Similarity"]:.3f})'
    )
    print(result["Text"])
    print()

Document 4 (0.649)
Accounts are temporarily locked after five failed login attempts. The account unlocks automatically after 30 minutes.

Document 1 (0.615)
If a user forgets their password, they can reset it using the Forgot Password link on the login page.

Document 5 (0.489)
Users can enable two-factor authentication from Settings > Security.

Document 3 (0.450)
If a file upload fails, check that the file is smaller than 10 MB and uses a supported format.

Document 2 (0.404)
Users can update their billing information from Settings > Billing > Payment Method.



## Using the retrieved information

The embedding model helped us rank the support cases.

Now I'll take the highest-ranked result and give it to the LLM along with the user's question.

In [7]:
top_result = results[0]
retrieved_context = top_result["Text"]

print("Question:")
print(query)

print("\nRetrieved context:")
print(retrieved_context)

print("\nSimilarity:")
print(round(top_result["Similarity"], 3))

Question:
I entered the wrong password several times and now I can't sign in.

Retrieved context:
Accounts are temporarily locked after five failed login attempts. The account unlocks automatically after 30 minutes.

Similarity:
0.649


## Generating the answer

The retrieval step found the relevant information. Now I'll pass that information to `llama3.2:3b` and ask it to answer the user's question.

The LLM isn't searching the knowledge base here. It only receives the context that was already retrieved.

In [8]:
prompt = f"""
Answer the user's question using only the context below.

Context:
{retrieved_context}

User question:
{query}

Keep the answer short and clear.
"""

response = ollama.chat(
    model="llama3.2:3b",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ]
)

answer = response["message"]["content"]

print(answer)

Your account is temporarily locked due to 5 failed login attempts. It will unlock automatically after 30 minutes.


## What happens if retrieval gets it wrong?

The LLM gave a good answer when it received the right context.

Now I'll give the same LLM an unrelated support case and keep everything else the same.

In [9]:
wrong_context = documents[1]

print("Question:")
print(query)

print("\nWrong context:")
print(wrong_context)

Question:
I entered the wrong password several times and now I can't sign in.

Wrong context:
Users can update their billing information from Settings > Billing > Payment Method.


In [10]:
wrong_prompt = f"""
Answer the user's question using only the context below.

Context:
{wrong_context}

User question:
{query}

If the context does not contain enough information to answer the question,
say that you don't have enough information.

Keep the answer short and clear.
"""

wrong_response = ollama.chat(
    model="llama3.2:3b",
    messages=[
        {
            "role": "user",
            "content": wrong_prompt
        }
    ]
)

print(wrong_response["message"]["content"])

I don't have enough information.


## What if I ask the LLM directly?

So far, the LLM received information retrieved from our small knowledge base.

Now I'll ask the same question directly without giving it any retrieved context.

In [11]:
direct_response = ollama.chat(
    model="llama3.2:3b",
    messages=[
        {
            "role": "user",
            "content": query
        }
    ]
)

direct_answer = direct_response["message"]["content"]

print(direct_answer)

I'm here to help you. If you've entered the wrong password multiple times and are now locked out of your account, don't worry, it's a common issue. Here are a few steps you can try to regain access:

1. **Check if you're using the correct password**: Double-check that you're entering the correct password. Make sure you're typing it in correctly, including any special characters or capital letters.
2. **Wait for the cooldown period**: Many password reset systems have a cooldown period after a certain number of incorrect attempts. Wait for this period to expire before trying again.
3. **Try a password reset**: If you're unable to sign in, try clicking on the "Forgot password" or "Reset password" link on the login page. This will send a password reset link to the email address associated with your account.
4. **Contact the account owner**: If you're trying to sign in to someone else's account, reach out to the account owner and ask for their password. Be respectful of their security and d

In [12]:
print("QUESTION")
print(query)

print("\n--- LLM ALONE ---")
print(direct_answer)

print("\n--- RETRIEVED CONTEXT ---")
print(retrieved_context)

print("\n--- RETRIEVAL + LLM ---")
print(answer)

QUESTION
I entered the wrong password several times and now I can't sign in.

--- LLM ALONE ---
I'm here to help you. If you've entered the wrong password multiple times and are now locked out of your account, don't worry, it's a common issue. Here are a few steps you can try to regain access:

1. **Check if you're using the correct password**: Double-check that you're entering the correct password. Make sure you're typing it in correctly, including any special characters or capital letters.
2. **Wait for the cooldown period**: Many password reset systems have a cooldown period after a certain number of incorrect attempts. Wait for this period to expire before trying again.
3. **Try a password reset**: If you're unable to sign in, try clicking on the "Forgot password" or "Reset password" link on the login page. This will send a password reset link to the email address associated with your account.
4. **Contact the account owner**: If you're trying to sign in to someone else's account, 

## What I found

The embedding model and the LLM were doing two different jobs.

The embedding model didn't answer the question. It helped find the support case that was closest to what the user was asking.

The LLM then used that retrieved information to write the final answer.

The LLM could also answer the question on its own using what it already knows. But that isn't the same as answering from the information in our knowledge base.

This became clearer when I gave it the wrong context. If retrieval brings back the wrong information, the generation step doesn't have the right information to work with.

## Takeaway

For me, the simple way to think about it is:

**Embeddings help find the context. The LLM uses that context to generate the answer.**

Both are useful, but they're solving different parts of the problem.